In [1]:
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium_models as models

# Load 2024 Data

In [5]:
import pandas as pd

# Only process 2024 season
year = 2024

# Read the data files
qb_df = pd.read_csv('new_qb_grades/cleaned_2023_qbs.csv')
weekly_fp_df = pd.read_csv('new_weekly_fp/2024.csv')
draft_rankings_df = pd.read_csv('new_draft_rankings/cleaned_2024_draft_rankings.csv')

# Define 2024 bye weeks for each team
bye_weeks_2024 = {
    'DET': 5, 'LAC': 5, 'PHI': 5, 'TEN': 5,
    'KAN': 6, 'LAR': 6, 'MIA': 6, 'MIN': 6,
    'ARI': 11, 'CAR': 11, 'NYG': 11, 'TAM': 11,
    'CLE': 10, 'GNB': 10, 'LVR': 10, 'SEA': 10, 'PIT': 9,
    'SFO': 9, 'CHI': 7, 'DAL': 7, 'HOU': 14, 'IND': 14,
    'JAX': 12, 'NWE': 14, 'ATL': 12, 'BAL': 14, 'BUF': 12,
    'CIN': 12, 'DEN': 14, 'NOR': 12, 'NYJ': 12, 'WAS': 14
}

# Create full schedule (weeks 1-18 for all players/teams)
all_weeks = range(1, 19)

# Get all unique player-team combinations from weekly data, but only for players in qb_df
valid_player_ids = set(qb_df['PFR_ID'].unique())
player_team_weeks = weekly_fp_df[weekly_fp_df['PFR_ID'].isin(valid_player_ids)].groupby(['PFR_ID', 'Team'])['Week'].agg(['min', 'max']).reset_index()
player_team_weeks.columns = ['PFR_ID', 'Team', 'first_week', 'last_week']

# Generate full schedule with correct team assignments per week
schedule_rows = []
for _, player_row in player_team_weeks.iterrows():
    player_id = player_row['PFR_ID']
    player_team = player_row['Team']
    first_week = player_row['first_week']
    last_week = player_row['last_week']

    # Only create rows for weeks this player was on this team
    for week in range(first_week, last_week + 1):
        is_bye = week == bye_weeks_2024.get(player_team, None)
        schedule_rows.append({
            'PFR_ID': player_id,
            'Week': week,
            'Team': player_team,
            'is_bye_week': is_bye
        })

schedule_df = pd.DataFrame(schedule_rows)

# Merge with actual weekly fantasy points
result_df = pd.merge(schedule_df, weekly_fp_df, on=['PFR_ID', 'Week', 'Team'], how='left')

# For rows where player didn't play (missing PPR) but it wasn't a bye week,
# fill in game context columns from another player who played in that game
game_context_cols = ['Day', 'game_num', 'Age', 'Home_Away', 'Opp', 'WINS', 'LOSSES', 'PF', 'PA',
                     'OVER', 'OFF', 'PASS', 'PBLK', 'RECV', 'RUN', 'RBLK', 'DEF',
                     'RDEF', 'TACK', 'PRSH', 'COV', 'WINS_opponent', 'LOSSES_opponent',
                     'PF_opponent', 'PA_opponent', 'OVER_opponent', 'OFF_opponent',
                     'PASS_opponent', 'PBLK_opponent', 'RECV_opponent', 'RUN_opponent',
                     'RBLK_opponent', 'DEF_opponent', 'RDEF_opponent', 'TACK_opponent',
                     'PRSH_opponent', 'COV_opponent']

# Identify rows that need filling (no PPR, not a bye week)
needs_filling = (result_df['PPR'].isna()) & (result_df['is_bye_week'] == False)

# For each row that needs filling, find a player from the same team/week with data
for idx in result_df[needs_filling].index:
    team = result_df.loc[idx, 'Team']
    week = result_df.loc[idx, 'Week']

    # Find any other row with same team and week that has data
    match = weekly_fp_df[(weekly_fp_df['Team'] == team) &
                         (weekly_fp_df['Week'] == week) &
                         (weekly_fp_df['PPR'].notna())]

    if len(match) > 0:
        # Use the first matching player's game context data (excluding Age and PPR)
        for col in game_context_cols:
            if col in match.columns and col != 'Age':  # Keep original player's age
                result_df.loc[idx, col] = match.iloc[0][col]

# Merge with QB grades
result_df = pd.merge(result_df, qb_df, on='PFR_ID', how='left', suffixes=('_TEAM', '_PLAYER'))

# Merge with draft rankings
result_df = pd.merge(result_df, draft_rankings_df, on='PFR_ID', how='left')

# Rename columns
result_df = result_df.rename(columns={'G#': 'game_num', '#G': 'games_played'})

# Clean RUN_PLAYER column
if 'RUN_PLAYER' in result_df.columns:
    result_df['RUN_PLAYER'] = result_df['RUN_PLAYER'].replace('-', '', regex=True)
    result_df['RUN_PLAYER'] = pd.to_numeric(result_df['RUN_PLAYER'], errors='coerce')

# Fill missing values for percentage columns
cols = ['P2S%', 'ADOT', 'ADJ%', 'DRP%']
existing_cols = [col for col in cols if col in result_df.columns]
if existing_cols:
    result_df[existing_cols] = result_df[existing_cols].fillna(result_df[existing_cols].mean())

# Fill missing ecr_adp_gap with 0
if 'ecr_adp_gap' in result_df.columns:
    result_df['ecr_adp_gap'] = result_df['ecr_adp_gap'].fillna(0)

# Add year column
result_df['YEAR'] = year

# Mark player status: played, bye, or missed game
result_df['player_status'] = 'missed_game'  # Default
result_df.loc[result_df['is_bye_week'] == True, 'player_status'] = 'bye_week'
result_df.loc[result_df['PPR'].notna(), 'player_status'] = 'played'
result_df['PPR'] = result_df['PPR'].fillna(0.0)
result_df['Age'] = result_df['Age'].ffill()
result_df['game_num'] = result_df['game_num'].ffill()

# Save to CSV
result_df.to_csv('gymnasium_qb_data_2024.csv', index=False)

In [6]:
import pandas as pd

# Only process 2024 season
year = 2024

# Read the data files
rb_df = pd.read_csv('new_rb_grades/cleaned_2023_rbs.csv')
weekly_fp_df = pd.read_csv('new_weekly_fp/2024.csv')
draft_rankings_df = pd.read_csv('new_draft_rankings/cleaned_2024_draft_rankings.csv')

# Define 2024 bye weeks for each team
bye_weeks_2024 = {
    'DET': 5, 'LAC': 5, 'PHI': 5, 'TEN': 5,
    'KAN': 6, 'LAR': 6, 'MIA': 6, 'MIN': 6,
    'ARI': 11, 'CAR': 11, 'NYG': 11, 'TAM': 11,
    'CLE': 10, 'GNB': 10, 'LVR': 10, 'SEA': 10, 'PIT': 9,
    'SFO': 9, 'CHI': 7, 'DAL': 7, 'HOU': 14, 'IND': 14,
    'JAX': 12, 'NWE': 14, 'ATL': 12, 'BAL': 14, 'BUF': 12,
    'CIN': 12, 'DEN': 14, 'NOR': 12, 'NYJ': 12, 'WAS': 14
}

# Create full schedule (weeks 1-18 for all players/teams)
all_weeks = range(1, 19)

# Get all unique player-team combinations from weekly data, but only for players in rb_df
valid_player_ids = set(rb_df['PFR_ID'].unique())
player_team_weeks = weekly_fp_df[weekly_fp_df['PFR_ID'].isin(valid_player_ids)].groupby(['PFR_ID', 'Team'])['Week'].agg(['min', 'max']).reset_index()
player_team_weeks.columns = ['PFR_ID', 'Team', 'first_week', 'last_week']

# Generate full schedule with correct team assignments per week
schedule_rows = []
for _, player_row in player_team_weeks.iterrows():
    player_id = player_row['PFR_ID']
    player_team = player_row['Team']
    first_week = player_row['first_week']
    last_week = player_row['last_week']

    # Only create rows for weeks this player was on this team
    for week in range(first_week, last_week + 1):
        is_bye = week == bye_weeks_2024.get(player_team, None)
        schedule_rows.append({
            'PFR_ID': player_id,
            'Week': week,
            'Team': player_team,
            'is_bye_week': is_bye
        })

schedule_df = pd.DataFrame(schedule_rows)

# Merge with actual weekly fantasy points
result_df = pd.merge(schedule_df, weekly_fp_df, on=['PFR_ID', 'Week', 'Team'], how='left')

# For rows where player didn't play (missing PPR) but it wasn't a bye week,
# fill in game context columns from another player who played in that game
game_context_cols = ['Day', 'game_num', 'Age', 'Home_Away', 'Opp', 'WINS', 'LOSSES', 'PF', 'PA',
                     'OVER', 'OFF', 'PASS', 'PBLK', 'RECV', 'RUN', 'RBLK', 'DEF',
                     'RDEF', 'TACK', 'PRSH', 'COV', 'WINS_opponent', 'LOSSES_opponent',
                     'PF_opponent', 'PA_opponent', 'OVER_opponent', 'OFF_opponent',
                     'PASS_opponent', 'PBLK_opponent', 'RECV_opponent', 'RUN_opponent',
                     'RBLK_opponent', 'DEF_opponent', 'RDEF_opponent', 'TACK_opponent',
                     'PRSH_opponent', 'COV_opponent']

# Identify rows that need filling (no PPR, not a bye week)
needs_filling = (result_df['PPR'].isna()) & (result_df['is_bye_week'] == False)

# For each row that needs filling, find a player from the same team/week with data
for idx in result_df[needs_filling].index:
    team = result_df.loc[idx, 'Team']
    week = result_df.loc[idx, 'Week']

    # Find any other row with same team and week that has data
    match = weekly_fp_df[(weekly_fp_df['Team'] == team) &
                         (weekly_fp_df['Week'] == week) &
                         (weekly_fp_df['PPR'].notna())]

    if len(match) > 0:
        # Use the first matching player's game context data (excluding Age and PPR)
        for col in game_context_cols:
            if col in match.columns and col != 'Age':  # Keep original player's age
                result_df.loc[idx, col] = match.iloc[0][col]

# Merge with RB grades
result_df = pd.merge(result_df, rb_df, on='PFR_ID', how='left', suffixes=('_TEAM', '_PLAYER'))

# Merge with draft rankings
result_df = pd.merge(result_df, draft_rankings_df, on='PFR_ID', how='left')

# Rename columns
result_df = result_df.rename(columns={'G#': 'game_num', '#G': 'games_played',
                                      'FUM.1': 'FUMBLE_GRADE_PLAYER',
                                      'YDS.1': 'RECV_YARDS'})

# Fill missing values for RB-specific percentage columns
cols = ['Y/RR', 'BAY%', 'PBLK_PLAYER', 'RECV_PLAYER', 'RBLK_PLAYER']
existing_cols = [col for col in cols if col in result_df.columns]
if existing_cols:
    result_df[existing_cols] = result_df[existing_cols].fillna(result_df[existing_cols].mean())

# Fill missing ecr_adp_gap with 0
if 'ecr_adp_gap' in result_df.columns:
    result_df['ecr_adp_gap'] = result_df['ecr_adp_gap'].fillna(0)

# Add year column
result_df['YEAR'] = year

# Mark player status: played, bye, or missed game
result_df['player_status'] = 'missed_game'  # Default
result_df.loc[result_df['is_bye_week'] == True, 'player_status'] = 'bye_week'
result_df.loc[result_df['PPR'].notna(), 'player_status'] = 'played'

# Fill remaining NA values
result_df['PPR'] = result_df['PPR'].fillna(0.0)
result_df['Age'] = result_df['Age'].ffill()
result_df['game_num'] = result_df['game_num'].ffill()

# Save to CSV
result_df.to_csv('gymnasium_rb_data_2024.csv', index=False)

In [7]:
import pandas as pd

# Only process 2024 season
year = 2024

# Read the data files
wr_te_df = pd.read_csv('new_wr_and_te_grades/cleaned_2023_wr_and_tes.csv')
weekly_fp_df = pd.read_csv('new_weekly_fp/2024.csv')
draft_rankings_df = pd.read_csv('new_draft_rankings/cleaned_2024_draft_rankings.csv')

# Define 2024 bye weeks for each team
bye_weeks_2024 = {
    'DET': 5, 'LAC': 5, 'PHI': 5, 'TEN': 5,
    'KAN': 6, 'LAR': 6, 'MIA': 6, 'MIN': 6,
    'ARI': 11, 'CAR': 11, 'NYG': 11, 'TAM': 11,
    'CLE': 10, 'GNB': 10, 'LVR': 10, 'SEA': 10, 'PIT': 9,
    'SFO': 9, 'CHI': 7, 'DAL': 7, 'HOU': 14, 'IND': 14,
    'JAX': 12, 'NWE': 14, 'ATL': 12, 'BAL': 14, 'BUF': 12,
    'CIN': 12, 'DEN': 14, 'NOR': 12, 'NYJ': 12, 'WAS': 14
}

# Create full schedule (weeks 1-18 for all players/teams)
all_weeks = range(1, 19)

# Get all unique player-team combinations from weekly data, but only for players in wr_te_df
valid_player_ids = set(wr_te_df['PFR_ID'].unique())
player_team_weeks = weekly_fp_df[weekly_fp_df['PFR_ID'].isin(valid_player_ids)].groupby(['PFR_ID', 'Team'])['Week'].agg(['min', 'max']).reset_index()
player_team_weeks.columns = ['PFR_ID', 'Team', 'first_week', 'last_week']

# Generate full schedule with correct team assignments per week
schedule_rows = []
for _, player_row in player_team_weeks.iterrows():
    player_id = player_row['PFR_ID']
    player_team = player_row['Team']
    first_week = player_row['first_week']
    last_week = player_row['last_week']

    # Only create rows for weeks this player was on this team
    for week in range(first_week, last_week + 1):
        is_bye = week == bye_weeks_2024.get(player_team, None)
        schedule_rows.append({
            'PFR_ID': player_id,
            'Week': week,
            'Team': player_team,
            'is_bye_week': is_bye
        })

schedule_df = pd.DataFrame(schedule_rows)

# Merge with actual weekly fantasy points
result_df = pd.merge(schedule_df, weekly_fp_df, on=['PFR_ID', 'Week', 'Team'], how='left')

# For rows where player didn't play (missing PPR) but it wasn't a bye week,
# fill in game context columns from another player who played in that game
game_context_cols = ['Day', 'game_num', 'Age', 'Home_Away', 'Opp', 'WINS', 'LOSSES', 'PF', 'PA',
                     'OVER', 'OFF', 'PASS', 'PBLK', 'RECV', 'RUN', 'RBLK', 'DEF',
                     'RDEF', 'TACK', 'PRSH', 'COV', 'WINS_opponent', 'LOSSES_opponent',
                     'PF_opponent', 'PA_opponent', 'OVER_opponent', 'OFF_opponent',
                     'PASS_opponent', 'PBLK_opponent', 'RECV_opponent', 'RUN_opponent',
                     'RBLK_opponent', 'DEF_opponent', 'RDEF_opponent', 'TACK_opponent',
                     'PRSH_opponent', 'COV_opponent']

# Identify rows that need filling (no PPR, not a bye week)
needs_filling = (result_df['PPR'].isna()) & (result_df['is_bye_week'] == False)

# For each row that needs filling, find a player from the same team/week with data
for idx in result_df[needs_filling].index:
    team = result_df.loc[idx, 'Team']
    week = result_df.loc[idx, 'Week']

    # Find any other row with same team and week that has data
    match = weekly_fp_df[(weekly_fp_df['Team'] == team) &
                         (weekly_fp_df['Week'] == week) &
                         (weekly_fp_df['PPR'].notna())]

    if len(match) > 0:
        # Use the first matching player's game context data (excluding Age and PPR)
        for col in game_context_cols:
            if col in match.columns and col != 'Age':  # Keep original player's age
                result_df.loc[idx, col] = match.iloc[0][col]

# Merge with WR/TE grades
result_df = pd.merge(result_df, wr_te_df, on='PFR_ID', how='left', suffixes=('_TEAM', '_PLAYER'))

# Merge with draft rankings
result_df = pd.merge(result_df, draft_rankings_df, on='PFR_ID', how='left')

# Rename columns
result_df = result_df.rename(columns={'G#': 'game_num', '#G': 'games_played',
                                      'FUM': 'FUMBLE_GRADE_PLAYER',
                                      'FUM.1': 'FUM_COUNT',
                                      'PASS_PLAYER': 'PASS_SNAPS',
                                      'RECV.1': 'RECV_SNAPS',
                                      'PBLK.1': 'PBLK_SNAPS'})

# Fill missing values for WR/TE-specific columns
cols = ['CTC%', 'DRP%', 'YAC/REC', 'PBLK_PLAYER', 'FUMBLE_GRADE_PLAYER',
        'DROP', 'Y/REC']
existing_cols = [col for col in cols if col in result_df.columns]
if existing_cols:
    result_df[existing_cols] = result_df[existing_cols].fillna(result_df[existing_cols].mean())

# Fill missing ecr_adp_gap with 0
if 'ecr_adp_gap' in result_df.columns:
    result_df['ecr_adp_gap'] = result_df['ecr_adp_gap'].fillna(0)

# Add year column
result_df['YEAR'] = year

# Mark player status: played, bye, or missed game
result_df['player_status'] = 'missed_game'  # Default
result_df.loc[result_df['is_bye_week'] == True, 'player_status'] = 'bye_week'
result_df.loc[result_df['PPR'].notna(), 'player_status'] = 'played'

# Fill remaining NA values
result_df['PPR'] = result_df['PPR'].fillna(0.0)
result_df['Age'] = result_df['Age'].ffill()
result_df['game_num'] = result_df['game_num'].ffill()

# Save to CSV
result_df.to_csv('gymnasium_wr_and_te_data_2024.csv', index=False)

# Create Playable Brains

In [2]:
qb_models = [models.QbKernelRidge]
rb_models = [models.RbKernelRidge]
wr_te_models = [models.WrTeKernelRidge]
models_columns = []

brain_1 = [qb_models[0], rb_models[0], wr_te_models[0]]

In [3]:
df = pd.read_csv("gymnasium_qb_data_2024.csv")
temp_model = brain_1[0]()
temp_model.predict(df)

,PFR_ID,Week,Prediction
0,AlleJo02,1,22.989164
1,AlleJo02,2,22.428538
2,AlleJo02,3,20.850535
3,AlleJo02,4,21.550326
4,AlleJo02,5,21.341507
...,...,...,...
636,YounBr01,14,16.386613
637,YounBr01,15,16.312086
638,YounBr01,16,17.831481
639,YounBr01,17,15.901900


In [4]:
df = pd.read_csv("gymnasium_rb_data_2024.csv")
temp_model = brain_1[1]()
temp_model.predict(df)

,PFR_ID,Week,Prediction
0,AbduAm00,2,2.255217
1,AbduAm00,3,2.465196
2,AbduAm00,4,2.380990
3,AbduAm00,5,2.321885
4,AbduAm00,6,2.382375
...,...,...,...
1340,WilsJe01,13,4.230963
1341,WilsJe01,14,4.060165
1342,WilsJe01,15,4.218240
1343,WilsJe01,16,3.226983


In [5]:
df = pd.read_csv("gymnasium_wr_and_te_data_2024.csv")
temp_model = brain_1[2]()
temp_model.predict(df)

,PFR_ID,Week,Prediction
0,AdamDa01,1,14.673980
1,AdamDa01,2,14.304546
2,AdamDa01,3,14.550405
3,AdamDa01,7,14.412770
4,AdamDa01,8,14.777344
...,...,...,...
3329,ZaccOl01,14,0.000000
3137,ZaccOl01,15,5.845186
3138,ZaccOl01,16,6.041956
3139,ZaccOl01,17,5.956674
